In [ ]:
!pip install torchmultimodal-nightly

In [ ]:
import os
import json
import torch
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms # Used for image transformations
from transformers import BertTokenizer # Used for text tokenization
from torchmultimodal.models.flava.model import flava_model_for_classification
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report

# 1. Configuration and Data Paths
train_json = "/kaggle/input/jsonfiles/multi_train.json"
val_json = "/kaggle/input/jsonfiles/multi_dev.json"
test_json = "/kaggle/input/jsonfiles/multi_test.json"
img_dir = "/kaggle/input/images/3k_image"
model_save_path = "/kaggle/working/FLAVA_Sarcasm"

# Labels mapping
label2id = {"Non-sarcasm": 0, "Sarcasm": 1}
id2label = {0: "Non-sarcasm", 1: "Sarcasm"}
num_labels = len(label2id)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Dataset and Processor Setup for FLAVA
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

image_transform = transforms.Compose([
    transforms.Resize([224, 224]),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class SarcasmDataset(Dataset):
    def __init__(self, json_file, img_dir):
        with open(json_file, 'r', encoding='utf-8') as f:
            self.data = json.load(f)
        self.img_dir = img_dir

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        text = item['caption']
        image_path = os.path.join(self.img_dir, item['image'])
        image = Image.open(image_path).convert('RGB')
        label = label2id[item['label']]

        # Return raw image, text, and integer label.
        # Transformations and tokenization will happen in collate_fn.
        return {
            "image": image,
            "text": text,
            "labels": torch.tensor(label)
        }

# Instantiate datasets without a 'processor' argument
train_dataset = SarcasmDataset(train_json, img_dir)
val_dataset = SarcasmDataset(val_json, img_dir)
test_dataset = SarcasmDataset(test_json, img_dir)

# 3. Collate Function for FLAVA
def collate_fn(batch_list):
    # Process text using BertTokenizer
    texts = [item["text"] for item in batch_list]
    tokenized_texts = tokenizer(
        texts,
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=512
    )
    input_ids = tokenized_texts["input_ids"]
    attention_mask = tokenized_texts["attention_mask"]

    # Process images using torchvision.transforms
    images = [image_transform(item["image"]) for item in batch_list]
    pixel_values = torch.stack(images) # Stack all image tensors

    # Process labels
    labels = torch.stack([item["labels"] for item in batch_list])

    return {
        "text": input_ids,
        "attention_mask": attention_mask,
        "image": pixel_values,
        "labels": labels
    }

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn)

# 4. Model Initialization (FLAVA)
model = flava_model_for_classification(num_classes=num_labels).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

# 5. Training and Validation Loop
epochs = 5

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}")

    # Training phase
    model.train()
    total_loss = 0
    for idx, batch in enumerate(tqdm(train_loader, desc="Training")):
        optimizer.zero_grad()
        batch = {k: v.to(device) for k, v in batch.items()}

        # FLAVA's forward expects 'text' as input_ids, 'image' as image tensor, and 'labels'
        out = model(text=batch["text"], image=batch["image"], labels=batch["labels"])
        loss = out.loss
        total_loss += loss.item()

        loss.backward()
        optimizer.step()

    avg_train_loss = total_loss / len(train_loader)
    print(f"Train Loss: {avg_train_loss:.4f}")

    # Evaluation phase
    model.eval()
    val_loss = 0
    preds = []
    targets = []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validation"):
            batch = {k: v.to(device) for k, v in batch.items()}
            out = model(text=batch["text"], image=batch["image"], labels=batch["labels"])
            val_loss += out.loss.item()

            logits = out.logits
            predictions = torch.argmax(logits, dim=-1).cpu().numpy()
            labels = batch["labels"].cpu().numpy()

            preds.extend(predictions)
            targets.extend(labels)

    avg_val_loss = val_loss / len(val_loader)
    acc = accuracy_score(targets, preds)
    f1 = f1_score(targets, preds, average='macro')
    precision = precision_score(targets, preds, average='macro')
    recall = recall_score(targets, preds, average='macro')

    print(f"Val Loss: {avg_val_loss:.4f} | Accuracy: {acc:.4f} | F1: {f1:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f}")

# Save model
if not os.path.exists(model_save_path):
    os.makedirs(model_save_path)
torch.save(model.state_dict(), os.path.join(model_save_path, "flava_sarcasm_model.pt"))
tokenizer.save_pretrained(model_save_path) # Save the tokenizer used


Epoch 1


Training: 100%|██████████| 75/75 [01:10<00:00,  1.06it/s]


Train Loss: 0.6305


Validation: 100%|██████████| 25/25 [00:11<00:00,  2.17it/s]
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Val Loss: 0.6134 | Accuracy: 0.6700 | F1: 0.4012 | Precision: 0.3350 | Recall: 0.5000

Epoch 2


...

Training: 100%|██████████| 75/75 [01:10<00:00,  1.07it/s]


Train Loss: 0.5807


Validation: 100%|██████████| 25/25 [00:10<00:00,  2.36it/s]


Val Loss: 0.6143 | Accuracy: 0.6600 | F1: 0.5047 | Precision: 0.5700 | Recall: 0.5310


('/kaggle/working/FLAVA_Sarcasm/tokenizer_config.json',
 '/kaggle/working/FLAVA_Sarcasm/special_tokens_map.json',
 '/kaggle/working/FLAVA_Sarcasm/vocab.txt',
 '/kaggle/working/FLAVA_Sarcasm/added_tokens.json')

In [ ]:
# Test Evaluation
print("\n--- Testing on Test Set ---")
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        labels = batch["labels"]
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(text=batch["text"], image=batch["image"], labels=batch["labels"])
        preds = torch.argmax(outputs.logits, dim=1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

print(classification_report(all_labels, all_preds, target_names=list(label2id.keys()), digits=4))


--- Testing on Test Set ---


Testing: 100%|██████████| 25/25 [00:11<00:00,  2.18it/s]

              precision    recall  f1-score   support

 Non-sarcasm     0.6761    0.8815    0.7653       135
     Sarcasm     0.3333    0.1231    0.1798        65

    accuracy                         0.6350       200
   macro avg     0.5047    0.5023    0.4725       200
weighted avg     0.5647    0.6350    0.5750       200

